In [1]:
import deeplabcut
import os
from deeplabcut.modelzoo import build_weight_init
import shutil
from modules.dlc_utils import set_transform_prob

project_path = 'projects/rat_pose'
config_path = os.path.join(project_path, "config.yaml")

Loading DLC 3.0.0rc13...


c:\Users\jiefei\anaconda3\envs\DEEPLABCUT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 475 frames with default augmentations
# shuttle=1

# 665 frames with customized augmentations
# shuttle=3

# 665 frames with default augmentations
# shuttle=4


# 1326 frames with default augmentations
# shuffle = 5

# Trained with superanimal_quadruped rtmpose_s + top-down detector
# shuffle = 6

# Trained with superanimal_quadruped rtmpose_s + top-down detector + unfreeze bn stats + default detector training epochs
shuffle = 7

In [3]:
from dlclibrary.dlcmodelzoo.modelzoo_download import _load_model_names
# dict to json text
from json import dumps
model_names = _load_model_names()
# for human readability
print(dumps(model_names, indent=4))

{
    "full_human": "mwmathis/DeepLabCutModelZoo-DLC_human_fullbody_resnet_101/DLC_human_dancing_resnet_101_iteration-0_shuffle-1.tar.gz",
    "full_cat": "AlexEMG/DeepLabCutModelZoo-cat/DLC_Cat_resnet_50_iteration-0_shuffle-0.tar.gz",
    "full_dog": "AlexEMG/DeepLabCutModelZoo-dog/DLC_Dog_resnet_50_iteration-0_shuffle-0.tar.gz",
    "primate_face": "mwmathis/DeepLabCutModelZoo-primate_face/DLC_primate_face_resnet_50_iteration-1_shuffle-1.tar.gz",
    "mouse_pupil_vclose": "mwmathis/DeepLabCutModelZoo-mouse_pupil_vclose/DLC_mouse_pupil_vclose_resnet_50_iteration-0_shuffle-1.tar.gz",
    "horse_sideview": "mwmathis/DeepLabCutModelZoo-horse_sideview/DLC_Horses_resnet_50_iteration-1_shuffle-1.tar.gz",
    "full_macaque": "mwmathis/DeepLabCutModelZoo-macaque_full/DLC_macaque_full_resnet50.tar.gz",
    "superanimal_topviewmouse_dlcrnet": "mwmathis/DeepLabCutModelZoo-SuperAnimal-TopViewMouse/DLC_ma_supertopview5k_resnet_50_iteration-0_shuffle-1.tar.gz",
    "superanimal_quadruped_dlcrnet": 

# initialize model weight

In [4]:
from deeplabcut.pose_estimation_pytorch import available_models
print(available_models())

['animaltokenpose_base', 'cspnext_m', 'cspnext_s', 'cspnext_x', 'ctd_coam_w32', 'ctd_coam_w48', 'ctd_coam_w48_human', 'ctd_prenet_hrnet_w32', 'ctd_prenet_hrnet_w48', 'ctd_prenet_rtmpose_m', 'ctd_prenet_rtmpose_s', 'ctd_prenet_rtmpose_x', 'ctd_prenet_rtmpose_x_human', 'dekr_w18', 'dekr_w32', 'dekr_w48', 'dlcrnet_stride16_ms5', 'dlcrnet_stride32_ms5', 'hrnet_w18', 'hrnet_w32', 'hrnet_w48', 'resnet_101', 'resnet_50', 'rtmpose_m', 'rtmpose_s', 'rtmpose_x', 'top_down_cspnext_m', 'top_down_cspnext_s', 'top_down_cspnext_x', 'top_down_hrnet_w18', 'top_down_hrnet_w32', 'top_down_hrnet_w48', 'top_down_resnet_101', 'top_down_resnet_50']


In [5]:
# superanimal_name = 'superanimal_mouse'
superanimal_name = 'superanimal_quadruped'
model_name = "rtmpose_s"
# detector_name="fasterrcnn_resnet50_fpn_v2"
detector_name = None
weight_init = build_weight_init(
            cfg = config_path,
            super_animal= superanimal_name,
            model_name=model_name,
            detector_name=detector_name,
            with_decoder=False
)


# Create training data

In [6]:
## delete `training-datasets` folder
path1 = os.path.join(project_path, 'training-datasets/iteration-0/UnaugmentedDataSet_Sleap_Rat_testOct2')
if os.path.exists(path1):
    name_contain = f"shuffle{shuffle}"
    # delete everything that contains `shuffle{shuffle}` in the name
    for item in os.listdir(path1):
        if name_contain in item:
            os.remove(os.path.join(path1, item))

path2 = os.path.join(project_path, f'dlc-models-pytorch/iteration-0/Sleap_Rat_testOct2-trainset95shuffle{shuffle}')
if os.path.exists(path2):
    shutil.rmtree(path2)
    
dt = deeplabcut.create_training_dataset(
    config_path, 
    Shuffles=[shuffle],    
    weight_init=weight_init, 
    net_type=model_name,  
    detector_type=detector_name,
    userfeedback=False)

F:\code\pose_track\projects\rat_pose\labeled-data\RAT 11 FR1\CollectedData_rats.h5  not found (perhaps not annotated).


# replace data augmentation parameters

In [7]:
from deeplabcut.core.config import read_config_as_dict
import deeplabcut.pose_estimation_pytorch as dlc_torch
import yaml

loader = dlc_torch.DLCLoader(
    config=config_path,  
    trainset_index=0,
    shuffle=shuffle,
)

# Get the pytorch config
pytorch_config_path = loader.model_folder / "pytorch_config.yaml"
model_cfg = read_config_as_dict(pytorch_config_path)
model_cfg

{'data': {'bbox_margin': 20,
  'colormode': 'RGB',
  'inference': {'normalize_images': True,
   'top_down_crop': {'width': 256, 'height': 256}},
  'train': {'affine': {'p': 0.5,
    'rotation': 30,
    'scaling': [1.0, 1.0],
    'translation': 0},
   'gaussian_noise': 12.75,
   'motion_blur': True,
   'normalize_images': True,
   'top_down_crop': {'width': 256, 'height': 256},
   'random_bbox_transform': {'shift_factor': 0.16,
    'shift_prob': 0.3,
    'scale_factor': [0.75, 1.25],
    'scale_prob': 1.0,
    'p': 1.0}}},
 'detector': {'data': {'colormode': 'RGB',
   'inference': {'normalize_images': True},
   'train': {'affine': {'p': 0.5,
     'rotation': 30,
     'scaling': [1.0, 1.0],
     'translation': 40},
    'collate': {'type': 'ResizeFromDataSizeCollate',
     'min_scale': 0.4,
     'max_scale': 1.0,
     'min_short_side': 128,
     'max_short_side': 1152,
     'multiple_of': 32,
     'to_square': False},
    'hflip': True,
    'normalize_images': True}},
  'device': 'auto',


In [8]:
# Set freeze_bn_stats=False for GPU training with large batch size
model_cfg["detector"]["model"]["freeze_bn_stats"] = False
# model_cfg["detector"]["train_settings"]["batch_size"] = 4

dlc_torch.config.write_config(pytorch_config_path, model_cfg)
model_cfg

{'data': {'bbox_margin': 20,
  'colormode': 'RGB',
  'inference': {'normalize_images': True,
   'top_down_crop': {'width': 256, 'height': 256}},
  'train': {'affine': {'p': 0.5,
    'rotation': 30,
    'scaling': [1.0, 1.0],
    'translation': 0},
   'gaussian_noise': 12.75,
   'motion_blur': True,
   'normalize_images': True,
   'top_down_crop': {'width': 256, 'height': 256},
   'random_bbox_transform': {'shift_factor': 0.16,
    'shift_prob': 0.3,
    'scale_factor': [0.75, 1.25],
    'scale_prob': 1.0,
    'p': 1.0}}},
 'detector': {'data': {'colormode': 'RGB',
   'inference': {'normalize_images': True},
   'train': {'affine': {'p': 0.5,
     'rotation': 30,
     'scaling': [1.0, 1.0],
     'translation': 40},
    'collate': {'type': 'ResizeFromDataSizeCollate',
     'min_scale': 0.4,
     'max_scale': 1.0,
     'min_short_side': 128,
     'max_short_side': 1152,
     'multiple_of': 32,
     'to_square': False},
    'hflip': True,
    'normalize_images': True}},
  'device': 'auto',


In [ ]:
# repace train data params with dropin params
with open("projects/rat_pose/train_dropin.yaml", 'r') as f:
    dropin_param = yaml.safe_load(f)
dropin_param

{'crop_sampling': {'width': 448,
  'height': 448,
  'max_shift': 0.1,
  'method': 'hybrid'},
 'normalize_images': True,
 'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32},
 'transform': [{'augmentation': 'Affine',
   'p': 1,
   'rotate': [-30, 30],
   'scale': [0.5, 1.25],
   'translate_px': [0, 0],
   'keep_ratio': True},
  {'augmentation': 'GaussNoise',
   'var_limit': [0, 162.5625],
   'mean': 0.0,
   'per_channel': True,
   'p': 1},
  {'MotionBlur': None, 'p': 1},
  {'augmentation': 'ImageCompression',
   'quality_lower': 20,
   'quality_upper': 90,
   'p': 1},
  {'augmentation': 'RandomSunFlare',
   'p': 1,
   'flare_roi': [0.0, 0.0, 1.0, 1],
   'angle_lower': 0.0,
   'angle_upper': 1.0,
   'num_flare_circles_lower': 1,
   'num_flare_circles_upper': 4,
   'src_radius': 40,
   'src_color': [255, 245, 230]},
  {'augmentation': 'RandomRain',
   'slant_lower': -1,
   'slant_upper': 1,
   'drop_length': 15,
   'drop_width': 4,
   'drop_color': [200, 200, 200],
   'blu

In [ ]:
model_cfg["data"]["train"] = dropin_param

model_cfg = set_transform_prob(model_cfg, prob=0.1)
n_trans = len(model_cfg["data"]["train"]['transform'])

# save
dlc_torch.config.write_config(pytorch_config_path,model_cfg)
model_cfg

{'data': {'bbox_margin': 20,
  'colormode': 'RGB',
  'inference': {'normalize_images': True,
   'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32}},
  'train': {'crop_sampling': {'width': 448,
    'height': 448,
    'max_shift': 0.1,
    'method': 'hybrid'},
   'normalize_images': True,
   'auto_padding': {'pad_width_divisor': 32, 'pad_height_divisor': 32},
   'transform': [{'augmentation': 'Affine',
     'p': 0.1,
     'rotate': [-30, 30],
     'scale': [0.5, 1.25],
     'translate_px': [0, 0],
     'keep_ratio': True},
    {'augmentation': 'GaussNoise',
     'var_limit': [0, 162.5625],
     'mean': 0.0,
     'per_channel': True,
     'p': 0.1},
    {'MotionBlur': None, 'p': 0.1},
    {'augmentation': 'ImageCompression',
     'quality_lower': 20,
     'quality_upper': 90,
     'p': 0.1},
    {'augmentation': 'RandomSunFlare',
     'p': 0.1,
     'flare_roi': [0.0, 0.0, 1.0, 1],
     'angle_lower': 0.0,
     'angle_upper': 1.0,
     'num_flare_circles_lower': 1,
     '

# Train

In [10]:
# delete all pt files 
import glob
pt_files = glob.glob(f'projects/rat_pose/dlc-models-pytorch/iteration-0/Sleap_Rat_testOct2-trainset95shuffle{shuffle}/train/*.pt')
for f in pt_files:
    os.remove(f)

In [9]:
deeplabcut.train_network(
    config_path,
    shuffle=shuffle,
    epochs=200,
    save_epochs=10,
    detector_epochs = 50,
    superanimal_name=superanimal_name,
    batch_size= 16,
    keepdeconvweights=False,
    device="cuda:0",
    superanimal_transfer_learning=True
    )

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [1.0, 1.0]
      translation: 0
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
    random_bbox_transform:
      shift_factor: 0.16
      shift_prob: 0.3
      scale_factor: [0.75, 1.25]
      scale_prob: 1.0
      p: 1.0
detector:
  data:
    colormode: RGB
    inference:
      normalize_images: True
    train:
      affine:
        p: 0.5
        rotation: 30
        scaling: [1.0, 1.0]
        translation: 40
      collate:
        type: ResizeFromDataSizeCollate
        min_scale: 0.4
        max_scale: 1.0
        min_short_side: 128
        max_short_side: 1152
        multiple_of: 32
        to_square: False
      hflip: True
      normalize_images: True
  device: auto
  